In [2]:
import os

from dotenv import load_dotenv

load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [24]:
import os

from groq import Groq

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

completion = client.chat.completions.create(
    model="llama3-70b-8192",
    messages=[
        {
            "role": "user",
            "content": "What is the mass of Earth plus the mass of Saturn and all of that times 2?",
        }
    ],
    temperature=0.7,
    max_tokens=1024,
    top_p=1,
    stream=True,
    stop=None,
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")


Let's calculate the mass of Earth plus the mass of Saturn, and then multiply the result by 2.

The mass of Earth is approximately 5.972 x 10^24 kilograms.

The mass of Saturn is approximately 5.684 x 10^26 kilograms.

Let's add the two masses together:

5.972 x 10^24 kg (Earth) + 5.684 x 10^26 kg (Saturn) = 5.684 x 10^26 kg + 5.972 x 10^24 kg

To add these numbers, we need to convert the mass of Earth to a value with the same exponent as the mass of Saturn:

5.972 x 10^24 kg = 0.5972 x 10^26 kg (approximately)

Now we can add the two masses:

0.5972 x 10^26 kg + 5.684 x 10^26 kg = 6.2812 x 10^26 kg

Finally, let's multiply the result by 2:

6.2812 x 10^26 kg x 2 = 12.5624 x 10^26 kg

So, the mass of Earth plus the mass of Saturn, multiplied by 2, is approximately 12.5624 x 10^26 kilograms.

In [26]:
class Agent:
    def __init__(self, client: Groq, system: str = "") -> None:
        self.client = client
        self.system = system
        self.messages: list = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message=""):
        if message:
            self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
            model="llama3-70b-8192", messages=self.messages
        )
        return completion.choices[0].message.content

In [5]:
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_planet_mass:
e.g. get_planet_mass: Earth
returns weight of the planet in kg

Example session:

Question: What is the mass of Earth times 2?
Thought: I need to find the mass of Earth
Action: get_planet_mass: Earth
PAUSE

You will be called again with this:

Observation: 5.972e24

Thought: I need to multiply this by 2
Action: calculate: 5.972e24 * 2
PAUSE

You will be called again with this:

Observation: 1,1944×10e25

If you have the answer, output it as the Answer.

Answer: The mass of Earth times 2 is 1,1944×10e25.

Now it's your turn:
""".strip()

In [7]:
def calculate(operation: str) -> float:
    return eval(operation)

def get_planet_mass(planet) -> float:
    match planet.lower():
        case "earth":
            return 5.972e24
        case "mars":
            return 6.39e23
        case "jupiter":
            return 1.898e27
        case "saturn":
            return 5.683e26
        case "uranus":
            return 8.681e25
        case "neptune":
            return 1.024e26
        case "mercury":
            return 3.285e23
        case "venus":
            return 4.867e24
        case _:
            return 0.0

In [39]:
import re

from print_text import print_text


def loop(max_iterations=20, query: str = "", verbose=False):

    agent = Agent(client=client, system=system_prompt)

    tools = ["calculate", "get_planet_mass"]

    next_prompt = query

    i = 0

    while i < max_iterations:
        i += 1
        result = agent(next_prompt)

        if verbose: print_text(result)

        if "PAUSE" in result and "Action" in result:
            action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
            chosen_tool = action[0][0]
            arg = action[0][1]

            if chosen_tool in tools:
                result_tool = eval(f"{chosen_tool}('{arg}')")
                next_prompt = f"Observation: {result_tool}"

            else:
                next_prompt = "Observation: Tool not found"

            if verbose:
                print_text(next_prompt)
            continue

        if "Answer" in result:
            print(result)
            break


loop(query="What is the mass of Earth plus the mass of Saturn and all of that times 2?", verbose=True)

Thought: I need to find the masses of Earth and Saturn and then perform the necessary calculations.
Action: get_planet_mass: Earth
PAUSE
Observation: 5.972e+24
Thought: Now I have the mass of Earth, I need to find the mass of Saturn.
Action: get_planet_mass: Saturn
PAUSE
Observation: 5.683e+26
Thought: I have the masses of both planets, now I need to add them together.
Action: calculate: 5.972e+24 + 5.683e+26
PAUSE
Observation: 5.74272e+26
Thought: Now I have the sum of the masses, I need to multiply it by 2.
Action: calculate: 5.74272e+26 * 2
PAUSE
Observation: 1.148544e+27
Answer: The mass of Earth plus the mass of Saturn and all of that times 2 is 1.148544e+27.
Answer: The mass of Earth plus the mass of Saturn and all of that times 2 is 1.148544e+27.


In [1]:
import base64

from groq import Groq


# Function to encode the image
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

# Path to your image
image_path = "/home/krish/NEO/stable-diffusion-xl_idk.jpg"

# Getting the base64 string
base64_image = encode_image(image_path)

client = Groq()

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "What's in this image?"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}",
                    },
                },
            ],
        }
    ],
    model="llava-v1.5-7b-4096-preview",
)

print(chat_completion.choices[0].message.content)

The image features a close-up of an illuminated glass ball or lamp, creating a captivating centerpiece. There are several white lightbulbs contained within the glass ball, casting a warm glow around it. The glittering, light-filled scene gives the impression of a beaded curtain or illuminated glass balls. The colors make the setting interesting and visually engaging.


The image features a close-up of an illuminated glass ball or lamp, creating a captivating centerpiece. There are several white lightbulbs contained within the glass ball, casting a warm glow around it. The glittering, light-filled scene gives the impression of a beaded curtain or illuminated glass balls. The colors make the setting interesting and visually engaging